# thư viện sử dụng

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Đường dẫn đến dữ liệu thô

In [2]:

train_path = Path(r"..\..\data\raw\train.csv")

In [3]:
test_path=Path(r"..\..\data\raw\test.csv")


# Đọc dữ liệu

In [4]:
df = pd.read_csv(train_path)

In [5]:
df_test=pd.read_csv(test_path)

In [6]:
print("Kích thước dữ liệu ban đầu:", df.shape)
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Số lượng missing values theo cột:")
print(missing)

Kích thước dữ liệu ban đầu: (1460, 81)
Số lượng missing values theo cột:
LotFrontage      259
Alley           1369
MasVnrType       872
MasVnrArea         8
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64


# 1. Điền các giá trị 0 (diện tích/số lượng = 0 khi không có tiện ích)

In [7]:
ZERO_FILL_COLS = [
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "BsmtFullBath",
    "BsmtHalfBath",
]

for col in ZERO_FILL_COLS:
    if col in df.columns:
        df[col] = df[col].fillna(0)

In [8]:
for col in ZERO_FILL_COLS:
    if col in df_test.columns:
        df_test[col] = df_test[col].fillna(0)

# 2. Điền các giá trị "None" (ngụ ý không có tiện ích tương ứng)

In [9]:
NA_SEMANTIC_MAP = {
    "PoolQC": "No Pool",
    "Alley": "No alley access",
    "Fence": "No Fence",
    "FireplaceQu": "No Fireplace",
    "GarageType": "No Garage",
    "GarageFinish": "No Garage",
    "GarageQual": "No Garage",
    "GarageCond": "No Garage",
    "BsmtQual": "No Basement",
    "BsmtCond": "No Basement",
    "BsmtExposure": "No Basement",
    "BsmtFinType1": "No Basement",
    "BsmtFinType2": "No Basement",
    "MiscFeature": "No Misc Feature",
}

for col, fill_value in NA_SEMANTIC_MAP.items():
    if col in df.columns:
        df[col] = df[col].fillna(fill_value)

In [10]:
for col, fill_value in NA_SEMANTIC_MAP.items():
    if col in df_test.columns:
        df_test[col] = df_test[col].fillna(fill_value)

# 3. Điền median cho các cột số thiếu thật sự

In [11]:
MEDIAN_FILL_COLS = [
    "LotFrontage",
]

for col in MEDIAN_FILL_COLS:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())


In [12]:
# Dùng median tính từ TRAIN, không tính riêng trên Kaggle test
for col in MEDIAN_FILL_COLS:
    if col in df_test.columns and col in df.columns:
        df_test[col] = df_test[col].fillna(df[col].median())

# 4. Điền mode cho các cột phân loại thiếu thật sự

In [13]:
MODE_FILL_COLS = [
    "MSZoning",
    "Utilities",
    "Exterior1st",
    "Exterior2nd",
    "Electrical",
    "KitchenQual",
    "Functional",
    "SaleType",
]

for col in MODE_FILL_COLS:
    if col in df.columns:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)

In [14]:
# Dùng mode từ TRAIN để xử lý test
for col in MODE_FILL_COLS:
    if col in df_test.columns and col in df.columns:
        modes = df[col].mode(dropna=True)
        if not modes.empty:
            df_test[col] = df_test[col].fillna(modes.iloc[0])

# 5. Xử lý ngoại lai (outliers)

## Ngoại lai: diện tích sống cực lớn nhưng giá bán bất thường rẻ

In [15]:
outlier_mask = (df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)
outlier_indices = df.index[outlier_mask]
print("Số lượng ngoại lai phát hiện (GrLivArea > 4000 & SalePrice < 300000):", len(outlier_indices))
print("Chỉ số ngoại lai:", outlier_indices.tolist())

Số lượng ngoại lai phát hiện (GrLivArea > 4000 & SalePrice < 300000): 2
Chỉ số ngoại lai: [523, 1298]


In [16]:
# Chỉ xóa outlier trong TRAIN; tuyệt đối giữ nguyên số dòng và Id của Kaggle test
df = df.drop(index=outlier_indices).reset_index(drop=True)
print("Kích thước train sau khi xóa outlier:", df.shape)
print("Kích thước Kaggle test (không xóa dòng):", df_test.shape)
print("Missing còn lại ở train:\n", df.isna().sum().loc[lambda s: s > 0])
print("Missing còn lại ở test:\n", df_test.isna().sum().loc[lambda s: s > 0])

Kích thước train sau khi xóa outlier: (1458, 81)
Kích thước Kaggle test (không xóa dòng): (1459, 80)
Missing còn lại ở train:
 MasVnrType     872
MasVnrArea       8
GarageYrBlt     81
dtype: int64
Missing còn lại ở test:
 MasVnrType     894
MasVnrArea      15
GarageYrBlt     78
dtype: int64


In [17]:
print(df.groupby("MasVnrType", dropna=False)["MasVnrArea"].describe())

            count        mean         std   min    25%    50%    75%     max
MasVnrType                                                                  
BrkCmn       15.0  247.666667  182.423239  41.0   91.5  192.0  381.5   621.0
BrkFace     445.0  259.008989  215.137888   0.0  114.0  202.0  336.0  1600.0
Stone       126.0  230.738095  159.259546   0.0  110.5  202.5  292.5   860.0
NaN         864.0    1.094907   18.569659   0.0    0.0    0.0    0.0   344.0


In [18]:
print(df.loc[
    df["GarageYrBlt"].isna(),
    ["GarageType", "GarageCars", "GarageArea"]
].head(10))

    GarageType  GarageCars  GarageArea
39   No Garage           0           0
48   No Garage           0           0
78   No Garage           0           0
88   No Garage           0           0
89   No Garage           0           0
99   No Garage           0           0
108  No Garage           0           0
125  No Garage           0           0
127  No Garage           0           0
140  No Garage           0           0


# xử lý tiếp các dữ liệu đặc biệt

In [19]:
mask_none = df["MasVnrArea"].eq(0)
mask_none_test = df_test["MasVnrArea"].eq(0)

# Không có dấu phẩy thừa ở cuối fillna (nếu có sẽ tạo tuple)
df.loc[mask_none, "MasVnrType"] = (
    df.loc[mask_none, "MasVnrType"].fillna("None")
)
df_test.loc[mask_none_test, "MasVnrType"] = (
    df_test.loc[mask_none_test, "MasVnrType"].fillna("None")
)

In [20]:
mask_unknown = (
    df["MasVnrType"].isna()
    & df["MasVnrArea"].gt(0)
)
mask_unknown_test = (
    df_test["MasVnrType"].isna()
    & df_test["MasVnrArea"].gt(0)
)

In [21]:
df.loc[mask_unknown, "MasVnrType"] = "Unknown"
df_test.loc[mask_unknown_test, "MasVnrType"] = "Unknown"

In [22]:
mask_both = df["MasVnrType"].isna() & df["MasVnrArea"].isna()
mask_both_test = df_test["MasVnrType"].isna() & df_test["MasVnrArea"].isna()
df.loc[mask_both, "MasVnrType"] = "Unknown"
df_test.loc[mask_both_test, "MasVnrType"] = "Unknown"

# Trường hợp còn thiếu loại veneer: giữ nhãn Unknown (không tự gán None)
df["MasVnrType"] = df["MasVnrType"].fillna("Unknown")
df_test["MasVnrType"] = df_test["MasVnrType"].fillna("Unknown")

In [23]:
# Cùng một giá trị median từ train
masvnr_median = df["MasVnrArea"].median()
df["MasVnrArea"] = df["MasVnrArea"].fillna(masvnr_median)
df_test["MasVnrArea"] = df_test["MasVnrArea"].fillna(masvnr_median)

# Tạo feature cho biết nhà có garage hay không

In [24]:
# Tạo cùng feature cho cả train và Kaggle test
for data in (df, df_test):
    data["HasGarage"] = (data["GarageCars"].fillna(0) > 0).astype(int)

# Nhà không có garage -> mã hóa năm xây bằng 0


In [25]:
# Nhà không có garage -> 0 là mã quy ước, không phải năm xây thật
for data in (df, df_test):
    no_garage = data["HasGarage"].eq(0) & data["GarageYrBlt"].isna()
    data.loc[no_garage, "GarageYrBlt"] = 0

# Nhà có garage nhưng thiếu năm xây -> median

In [26]:
garage_median = df.loc[
    df["HasGarage"].eq(1),
    "GarageYrBlt"
].median()

In [27]:
# Áp dụng median từ train cho cả train và test (nếu còn thiếu năm xây)
df["GarageYrBlt"] = df["GarageYrBlt"].fillna(garage_median)
df_test["GarageYrBlt"] = df_test["GarageYrBlt"].fillna(garage_median)

In [28]:
# Kiểm tra cả hai tập và đảm bảo Kaggle test giữ đúng Id, số dòng
train_missing = df.isna().sum()
test_missing = df_test.isna().sum()
print("Missing train còn lại:\n", train_missing[train_missing > 0])
print("Missing test còn lại:\n", test_missing[test_missing > 0])
assert df_test["Id"].equals(pd.read_csv(test_path)["Id"]), "Test Id/thứ tự dòng đã thay đổi!"
print("Test Id & row order: OK")

Missing train còn lại:
 Series([], dtype: int64)
Missing test còn lại:
 Series([], dtype: int64)
Test Id & row order: OK


In [29]:
# Xuất cả train và Kaggle test đã CLEANING (chưa scaling/encoding)
output_dir = Path("../../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(output_dir / "cleaned_train.csv", index=False)
df_test.to_csv(output_dir / "cleaned_test.csv", index=False)
df_test[["Id"]].to_csv(output_dir / "test_ids.csv", index=False)
print("Đã lưu:", output_dir.resolve())

Đã lưu: D:\DOAN\machine_learning\Ames\data\processed
